In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 5.1 Institutional Pattern Discovery with Unsupervised Learning
- No target variable — we **discover** structure
- K-Means clustering + PCA visualization
- Applied to higher education institutional questions
- Uses matplotlib/seaborn for static visualizations

## Supervised vs. Unsupervised
- **Supervised** (Modules 2-4): Predict a known target (DEPARTED)
- **Unsupervised** (this module): Find hidden groupings without a target
- Key question: "What natural patterns exist in our data?"

## The Provost's Question
> "We have 1,000 incoming first-year students. Can we identify **natural groupings** to tailor orientation and advising programs?"

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("Libraries loaded!")

## Case Study 1: Student Entry Segmentation

### Step 1: Generate Data

In [ ]:
n = 1000

entry_df = pd.DataFrame({
    'HS_GPA':             np.random.normal(3.2, 0.45, n).clip(2.0, 4.0),
    'HS_MATH_GPA':        np.random.normal(3.0, 0.55, n).clip(1.5, 4.0),
    'HS_ENGL_GPA':        np.random.normal(3.1, 0.50, n).clip(1.5, 4.0),
    'UNITS_ATTEMPTED_1':  np.random.choice([9, 12, 13, 14, 15, 16, 17, 18], n,
                                            p=[0.05, 0.15, 0.10, 0.15, 0.25, 0.15, 0.10, 0.05]),
    'FIRST_GEN':          np.random.binomial(1, 0.42, n),
    'PELL_ELIGIBLE':      np.random.binomial(1, 0.38, n),
    'DISTANCE_FROM_CAMPUS': np.random.exponential(25, n).clip(1, 200).round(1)
})

print(f"Dataset: {entry_df.shape[0]:,} students x {entry_df.shape[1]} variables")
entry_df.describe().round(2)

### Step 2: Scale the Data
- K-Means uses Euclidean distance
- Without scaling, variables with larger ranges dominate
- StandardScaler: mean=0, std=1

In [ ]:
scaler = StandardScaler()
entry_scaled = scaler.fit_transform(entry_df)

print("Before scaling:")
print(f"  HS_GPA range:    {entry_df['HS_GPA'].min():.1f} - {entry_df['HS_GPA'].max():.1f}")
print(f"  DISTANCE range:  {entry_df['DISTANCE_FROM_CAMPUS'].min():.1f} - {entry_df['DISTANCE_FROM_CAMPUS'].max():.1f}")
print(f"\nAfter scaling (mean ~ 0, std ~ 1):")
print(f"  HS_GPA range:    {entry_scaled[:, 0].min():.2f} - {entry_scaled[:, 0].max():.2f}")
print(f"  DISTANCE range:  {entry_scaled[:, 6].min():.2f} - {entry_scaled[:, 6].max():.2f}")

### Step 3: Find Optimal k
- **Elbow Method**: Plot inertia vs. k, look for the "bend"
- **Silhouette Score**: How similar points are to their own cluster vs. others

In [ ]:
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(entry_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(entry_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k = 4')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')
axes[1].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k = 4')
axes[1].legend()

plt.tight_layout()
plt.show()

### Step 4: Fit K-Means with k=4

In [ ]:
km_entry = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE)
entry_df['Cluster'] = km_entry.fit_predict(entry_scaled)

print(f"Silhouette Score: {silhouette_score(entry_scaled, entry_df['Cluster']):.3f}")
print(f"\nCluster Sizes:")
print(entry_df['Cluster'].value_counts().sort_index())

### Step 5: PCA for Visualization
- 7 variables -> can't plot directly
- PCA compresses to 2 dimensions that capture the most variance

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
entry_pca = pca.fit_transform(entry_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(entry_pca[:, 0], entry_pca[:, 1],
                     c=entry_df['Cluster'], cmap='Set2', alpha=0.6, s=30, edgecolors='w', linewidth=0.3)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Student Entry Segments (PCA Projection)')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

print(f"PC1 + PC2 explain {sum(pca.explained_variance_ratio_[:2]):.1%} of variance")

### Step 6: Cluster Profiling
- Most important step for institutional use
- Compute mean of each variable per cluster
- Interpret what makes each group distinctive

In [ ]:
profile = entry_df.groupby('Cluster').mean().round(2)
profile['Count'] = entry_df['Cluster'].value_counts().sort_index()

print("=" * 70)
print("CLUSTER PROFILES - Student Entry Segmentation")
print("=" * 70)
print(profile.to_string())

profile_z = (profile.drop(columns='Count') - profile.drop(columns='Count').mean()) / profile.drop(columns='Count').std()

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(profile_z, annot=profile.drop(columns='Count').values, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=1, ax=ax)
ax.set_title('Cluster Profiles (color = z-score, numbers = raw means)')
ax.set_ylabel('Cluster')
plt.tight_layout()
plt.show()

### Interpretation Example
| Cluster | Label | Key Characteristics | Suggested Intervention |
|:--------|:------|:-------------------|:----------------------|
| 0 | Prepared Commuters | High GPA, far from campus | Remote support resources |
| 1 | At-Risk First-Gen | Lower GPA, first-gen, Pell | Intensive advising |
| 2 | Strong Locals | High GPA, close to campus | Leadership opportunities |
| 3 | Mid-Range | Average across the board | Standard orientation |

## Case Study 4: Course Bottleneck Detection
> "Can we identify clusters of problematic courses — high DFW, low pass rates?"

In [ ]:
n_courses = 1000

course_df = pd.DataFrame({
    'ENROLLMENT':        np.random.lognormal(3.5, 0.5, n_courses).clip(15, 500).astype(int),
    'DFW_RATE':          np.random.beta(2, 7, n_courses).clip(0.01, 0.70).round(3),
    'PASS_RATE':         np.random.beta(7, 2, n_courses).clip(0.30, 0.99).round(3),
    'AVG_GPA':           np.random.normal(2.7, 0.5, n_courses).clip(0.5, 4.0).round(2),
    'REPEAT_RATE':       np.random.beta(1.5, 8, n_courses).clip(0.0, 0.50).round(3),
    'AVG_REPEAT_DELAY':  np.random.exponential(1.5, n_courses).clip(0.5, 6.0).round(1),
    'SECTION_SIZE':      np.random.choice([25, 30, 35, 40, 50, 60, 80, 100, 150, 200], n_courses),
    'PCT_FIRST_YEAR':    np.random.beta(3, 5, n_courses).round(3),
    'PCT_STEM':          np.random.beta(3, 4, n_courses).round(3),
    'INSTRUCTOR_RATING': np.random.normal(3.8, 0.6, n_courses).clip(1.0, 5.0).round(1),
    'PREREQUISITE_COUNT': np.random.choice([0, 1, 2, 3, 4, 5], n_courses, p=[0.15, 0.30, 0.25, 0.15, 0.10, 0.05]),
    'IS_GATEWAY':        np.random.binomial(1, 0.30, n_courses),
    'UNITS':             np.random.choice([1, 2, 3, 4, 5], n_courses, p=[0.05, 0.10, 0.50, 0.30, 0.05])
})

print(f"Dataset: {course_df.shape[0]:,} courses x {course_df.shape[1]} variables")

In [ ]:
scaler_c = StandardScaler()
course_scaled = scaler_c.fit_transform(course_df)

sils_c = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    sils_c.append(silhouette_score(course_scaled, km.fit_predict(course_scaled)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(2, 11), sils_c, 'go-', linewidth=2, markersize=8)
ax.set_xlabel('k'); ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette - Course Sections')
ax.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k = 4')
ax.legend(); plt.tight_layout(); plt.show()

km_course = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE)
course_df['Cluster'] = km_course.fit_predict(course_scaled)

pca_c = PCA(n_components=2, random_state=RANDOM_STATE)
course_pca = pca_c.fit_transform(course_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(course_pca[:, 0], course_pca[:, 1],
    c=course_df['Cluster'], cmap='Set2', alpha=0.6, s=30, edgecolors='w', linewidth=0.3)
ax.set_xlabel(f'PC1 ({pca_c.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_c.explained_variance_ratio_[1]:.1%})')
ax.set_title('Course Section Segments (PCA)')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout(); plt.show()

In [ ]:
profile_c = course_df.groupby('Cluster').mean().round(3)
profile_c['Count'] = course_df['Cluster'].value_counts().sort_index()
print("=" * 100)
print("CLUSTER PROFILES - Course Bottleneck Detection")
print("=" * 100)
print(profile_c.to_string())

profile_cz = (profile_c.drop(columns='Count') - profile_c.drop(columns='Count').mean()) / profile_c.drop(columns='Count').std()
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(profile_cz, annot=True, fmt='.2f', cmap='RdYlGn_r', center=0, linewidths=1, ax=ax)
ax.set_title('Course Cluster Profiles (z-scored)')
ax.set_ylabel('Cluster')
plt.tight_layout(); plt.show()

### Risk Prioritization Index

In [ ]:
course_df['RISK_INDEX'] = (
    0.4 * (course_df['DFW_RATE'] / course_df['DFW_RATE'].max()) +
    0.3 * (course_df['REPEAT_RATE'] / course_df['REPEAT_RATE'].max()) +
    0.3 * ((1 - course_df['PASS_RATE']) / (1 - course_df['PASS_RATE']).max())
).round(3)

top_risk = course_df.nlargest(20, 'RISK_INDEX')[
    ['ENROLLMENT', 'DFW_RATE', 'PASS_RATE', 'REPEAT_RATE', 'IS_GATEWAY', 'RISK_INDEX', 'Cluster']
]

print("=" * 80)
print("TOP 20 HIGHEST-RISK COURSE SECTIONS")
print("=" * 80)
print(top_risk.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
course_df.boxplot(column='RISK_INDEX', by='Cluster', ax=axes[0])
axes[0].set_title('Risk Index by Cluster')
axes[0].set_xlabel('Cluster'); axes[0].set_ylabel('Risk Index')

for c in sorted(course_df['Cluster'].unique()):
    axes[1].hist(course_df[course_df['Cluster'] == c]['RISK_INDEX'],
                 alpha=0.5, bins=20, label=f'Cluster {c}')
axes[1].set_xlabel('Risk Index'); axes[1].set_ylabel('Count')
axes[1].set_title('Risk Index Distribution by Cluster')
axes[1].legend()
plt.tight_layout(); plt.show()

## The Unsupervised Learning Pipeline
```
1. Scale data (StandardScaler)
2. Find optimal k (Elbow + Silhouette)
3. Fit K-Means
4. Visualize with PCA
5. Profile clusters (means + heatmap)
6. Interpret for stakeholders
```

## Key Takeaways
- Unsupervised learning discovers hidden structure without a target
- Always scale before K-Means
- Elbow + Silhouette to pick k
- PCA for visualization
- Cluster profiling is the most important deliverable
- **Module 5 Complete! Next: Module 6 (Special Topics - Optional)**